# Nextgen doğal sohbet + RAG LLM eğitimi (BPE subword)

Bu notebook, **doğal konuşan** decoder-only LLM'i eğitir (RAG bilgi parçalarıyla birlikte).

## Adımlar
1. **Çalışma Zamanı → Çalışma zamanı türünü değiştir → T4 GPU** seçin (eğitim çok hızlanır).
2. **En güvenlisi: Çalışma Zamanı → Tümünü çalıştır.** Aşağıdaki 3. hücre durup **dosya seçme penceresi** açar — dosyaları HEP BİRDEN seçin, gerisi otomatik sürer:
   `train_llm.py, llm.py, bpe.py, train_tokenizer.py, seqgen.py, naturalize.py, normalize.py, seq2seq.py, brain.py, corpus.py, transformer.py, intents.json, corpus.jsonl, tokenizer/bpe.json`
3. Elle sırayla basacaksanız min zorunlular: **Hücre 3** (dosya yükle) → **Hücre 5** (tokenizer hazırlığı) → **Hücre 6** (EĞİTİM) → **Hücre 7** (indir). Hücre 2 ve 4 isteğe bağlı kontrollerdir.
4. **Tokenizer**: `tokenizer/bpe.json` yüklüyse doğrudan kullanılır; yoksa `train_tokenizer.py` CPU'da ~2 dk içinde yeniden öğrenir (16K vocab).
5. En sondaki hücre `llm_model.json`'u bilgisayarına indirir → bilgisayarında `model/llm_model.json` olarak kaydet.

> Dosya seçmek uğraştırırsa alternatif: soldaki 📁 (Dosyalar) paneline dosyaları sürükle-bırak, `tokenizer/` klasörünü de panelden oluşturup `bpe.json`'u içine koy, ardından Hücre 5, 6 ve 7'yi çalıştır.

In [ ]:
import sys
sys.path.insert(0, '/content')
import torch
print('torch:', torch.__version__)
print('GPU :', 'Kullaniliyor (T4)' if torch.cuda.is_available() else 'YOK - üstteki menüden T4 GPU seçin')

In [ ]:
from google.colab import files
import os

print('Dosya sec penceresi acildi. Tum dosyalari HEP BIRDEN sec (Ctrl/Cmd + tik):')
print('  train_llm.py, llm.py, bpe.py, train_tokenizer.py')
print('  seqgen.py, naturalize.py, normalize.py, seq2seq.py')
print('  brain.py, corpus.py, transformer.py')
print('  intents.json, corpus.jsonl, tokenizer/bpe.json')
print('  knowledge_map.jsonl')
uploaded = files.upload()

required = ['train_llm.py','llm.py','bpe.py','train_tokenizer.py',
            'seqgen.py','naturalize.py','normalize.py',
            'seq2seq.py','brain.py','corpus.py','transformer.py',
            'intents.json','corpus.jsonl','knowledge_map.jsonl','tokenizer/bpe.json']
missing = [f for f in required if not os.path.exists('/content/' + f)]
if missing:
    print('EKSIK DOSYALAR (bunlari Files paneline surukle-birak):', missing)
else:
    print('OK - tum dosyalar yerinde. Hucr 4 calistirmaya gec.')

In [ ]:
import os
missing = [f for f in ['train_llm.py','llm.py','bpe.py','train_tokenizer.py',
                       'seqgen.py','naturalize.py','normalize.py',
                       'seq2seq.py','brain.py','corpus.py','transformer.py',
                       'intents.json','corpus.jsonl','knowledge_map.jsonl','tokenizer/bpe.json']
            if not os.path.exists('/content/' + f)]
print('Eksik:', missing) if missing else print('Hazir - egitim basliyor')

In [ ]:
import os
import sys
sys.path.insert(0, '/content')
tok = '/content/tokenizer/bpe.json'
if not os.path.exists(tok):
    print('Tokenizer yok -> CPU'da yeniden egitiliyor (16K vocab)...')
    os.makedirs('/content/tokenizer', exist_ok=True)
    !python train_tokenizer.py --vocab-size 16000 --max-texts 20000 --progress | tail -6
else:
    from bpe import BPETokenizer
    t = BPETokenizer.load(tok)
    print('Tokenizer hazir:', tok, '(vocab=' + str(len(t)) + ')')

In [ ]:
%%time
!python train_llm.py --rag --kb-map knowledge_map.jsonl --epochs 250

In [ ]:
import os
p = '/content/llm_model.json'
if os.path.exists(p):
    from google.colab import files
    print('Lutfen bu dosyayi bilgisayarinda model/ klasorune kaydet:', p, f'({os.path.getsize(p)//1024} KB)')
    files.download(p)
else:
    print('llm_model.json henuz yok - egitim bitmedi veya hata oldu. Yukaridaki ciktiyi kontrol et.')